Containers

Sequential :a container that applies layer one after another 

In [1]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 2)
)

print(model)

Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=2, bias=True)
)


ModuleList

Stores modules in a Python-like list.(avlot of layers )
it's like you make a 3 layers and you give them a name to easy using 
(a piece of nn)(this partie od NN will be stored )

In [2]:
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.ModuleList([
            nn.Linear(10, 20),
            nn.Linear(20, 30),
            nn.Linear(30, 2)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = Net()

a normal list it's bad :
if we do 
    `
    self.layers = [
        nn.linear(10,20),
        nn.Linear(20,30)
    ]
    `
pytorch will not register these layer
so we use ModuleList

 ModuleDict

Stores modules in a dictionary.
it's useful when You want to choose layers by name

In [3]:
import torch.nn as nn 
layers = nn.ModuleDict({
    "input": nn.Linear(10, 20),
    "hidden": nn.Linear(20, 30),
    "output": nn.Linear(30, 2)
})

x = torch.randn(1, 10)
x = layers["input"](x)
x = layers["hidden"](x)
x = layers["output"](x)
print(x)

tensor([[-0.3623, -0.3462]], grad_fn=<AddmmBackward0>)


ParameterList
Stores trainable parameters in a list.
nn.parameter Creates a trainable tensor.
PyTorch optimizer automatically updates it.

In [8]:
import torch 
import torch.nn as nn 

class MyNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(10, 20)),
            nn.Parameter(torch.randn(20, 30)),
            nn.Parameter(torch.randn(30, 2))
        ])  
model = MyNet()

In [9]:
for name, param in model.named_parameters():
    print(name, param.shape)

weights.0 torch.Size([10, 20])
weights.1 torch.Size([20, 30])
weights.2 torch.Size([30, 2])


ParameterDict

Stores trainable parameters in a dictionary.

In [37]:
import torch.nn as nn
import torch

class MyNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.params = nn.ParameterDict({
            "input": nn.Parameter(torch.randn(10, 20)),
            "hidden": nn.Parameter(torch.randn(20, 30)),
            "output": nn.Parameter(torch.randn(30, 2))
        })
model = MyNet()


In [38]:
print(model.params["input"])

Parameter containing:
tensor([[ 0.0867,  0.8423,  0.1159,  1.4914, -1.0657,  0.4154,  0.2676, -0.5121,
          1.0509, -0.3135, -0.8157, -0.8246, -1.1843,  1.6365,  3.2800, -0.9349,
         -1.7931, -1.5026, -0.5849, -0.3072],
        [-0.3855,  1.3060, -0.5874, -0.5537, -0.5454,  0.6988,  0.2585, -0.8158,
         -0.1555,  0.2162, -0.6567, -0.6492, -0.5008,  1.4591,  1.2307,  0.6938,
          0.2690,  0.5505, -0.3936,  0.2334],
        [-0.5354, -0.2476,  1.6213, -1.2698, -0.2276,  0.0683,  1.6264, -1.4903,
          0.3644, -0.5330,  0.6367, -0.6892, -0.3265, -1.7706, -0.1440, -1.9112,
          0.7978, -0.8837, -0.1803,  0.7898],
        [ 1.2094,  0.8829, -1.7761,  0.3174, -1.3068,  1.4192, -0.1161,  1.0976,
         -0.0516,  0.3118,  0.3779,  0.6084, -1.9797, -0.0731,  1.4422,  0.0645,
          0.8049,  0.4253, -1.3748, -1.3590],
        [ 0.9071,  1.2942,  0.3514,  3.4411,  0.1462,  1.2458,  0.0852, -0.6112,
          0.0674, -0.1240, -1.4243,  0.2023,  0.5464, -0.8406,  0

A hook is a function that PyTorch automatically calls at a specific moment during the execution of a neural network.

Think of a hook like a security camera attached to a layer:

Before a layer executes → Pre Hook
After a layer executes → Forward Hook
During backpropagation → Backward Hook

The hook allows you to inspect, log, or modify data without changing the layer's code.

normal execution 
suppose we have :

In [30]:
import torch
import torch.nn as nn 

layer = nn.Linear(3, 2)
print(layer.weight)
print(layer.bias)
x = torch.tensor([[1.0, 2.0, 3.0]])
y = layer(x)
print(y)

Parameter containing:
tensor([[ 0.2259,  0.3402, -0.2518],
        [-0.4762,  0.1482, -0.3376]], requires_grad=True)
Parameter containing:
tensor([-0.0799,  0.5259], requires_grad=True)
tensor([[ 0.0710, -0.6668]], grad_fn=<AddmmBackward0>)


exection 
input x ------> linear layer ----> output y

1.forward pre hook
A Forward Pre Hook runs before the layer computes its output.

In [31]:
import torch
import torch.nn as nn

def pre_hook(module, input):
    print("before layer execution")
    print("input", input)

layer = nn.Linear(3, 2)
print(layer.weight)
print(layer.bias)
print("before hook")
layer.register_forward_pre_hook(pre_hook) # register a pre-hook to the layer
x = torch.tensor([[1.0, 2.0, 3.0]])
y = layer(x)
print(y)


Parameter containing:
tensor([[ 0.0164,  0.4980,  0.2344],
        [ 0.3813, -0.1157,  0.5369]], requires_grad=True)
Parameter containing:
tensor([ 0.5201, -0.0625], requires_grad=True)
before hook
before layer execution
input (tensor([[1., 2., 3.]]),)
tensor([[2.2356, 1.6981]], grad_fn=<AddmmBackward0>)


so what is happen :
before layer exection 
input(tensor([[1.,2.,3.]]))

pytorch does :
1.call pre_hook()
2.execute Linear layer 
3.return output

Input
  ↓
Pre Hook
  ↓
Linear Layer
  ↓
Output

2.Forward Hook
A Forward Hook runs after the layer computes its output

In [32]:
import torch
import torch.nn as nn

def forward_hook(module, input, output):
    print("after layer execution")
    print("input", input)
    print("output", output)

layer = nn.Linear(3, 2)
print(layer.weight)
print(layer.bias)
print("before hook")
layer.register_forward_hook(forward_hook) # register a forward hook to the layer
x = torch.tensor([[1.0, 2.0, 3.0]])
y = layer(x)
print(y)

Parameter containing:
tensor([[-0.0778,  0.1206, -0.0923],
        [-0.4192,  0.5373, -0.0869]], requires_grad=True)
Parameter containing:
tensor([ 0.3009, -0.5368], requires_grad=True)
before hook
after layer execution
input (tensor([[1., 2., 3.]]),)
output tensor([[ 0.1875, -0.1420]], grad_fn=<AddmmBackward0>)
tensor([[ 0.1875, -0.1420]], grad_fn=<AddmmBackward0>)


Input
  ↓
Linear Layer
  ↓
Forward Hook
  ↓(it's execte after the calculation is over )
Output


remarque :the input parameter is actually a tuple containing the inputs to the layer:

Hook Arguements 
module :the layer itself 


In [35]:
print(model)

MyNet(
  (params): ParameterDict(
      (hidden): Parameter containing: [torch.FloatTensor of size 20x30]
      (input): Parameter containing: [torch.FloatTensor of size 10x20]
      (output): Parameter containing: [torch.FloatTensor of size 30x2]
  )
)


input recevived by the layer 

In [42]:
print(model)    

MyNet(
  (params): ParameterDict(
      (hidden): Parameter containing: [torch.FloatTensor of size 20x30]
      (input): Parameter containing: [torch.FloatTensor of size 10x20]
      (output): Parameter containing: [torch.FloatTensor of size 30x2]
  )
)
